# RAG with Small Models — GPU Version

**Runtime required: T4 GPU** (`Runtime → Change runtime type → T4 GPU`)

Same pipeline as the CPU version but:
- Embedding model runs on GPU
- Qwen2.5-3B-Instruct GGUF loaded with full CUDA support
- All layers offloaded to GPU (`n_gpu_layers=-1`)

Expected generation speed: ~50-80 tokens/second vs ~3 tokens/second on CPU.

## Step 0 — Install Dependencies (CUDA version of llama-cpp-python)

In [ ]:
# Pin numpy FIRST before anything else installs a conflicting version
!pip install numpy==1.26.4 --quiet

# Standard dependencies
!pip install wikipedia-api sentence-transformers lancedb pyarrow pandas tqdm --quiet

# llama-cpp-python compiled with CUDA support — this is what makes GPU inference work
# Without this, the model silently falls back to CPU even in a GPU runtime
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 \
  --force-reinstall --quiet

print("Done.")

## Step 1 — Verify GPU is Available

In [ ]:
import torch

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM total:    {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("No GPU found. Switch runtime to T4 GPU and restart.")

## Step 2 — Build the Knowledge Base

In [ ]:
import wikipediaapi

wiki = wikipediaapi.Wikipedia(user_agent='RAG-Demo/1.0', language='en')

article_titles = [
    "Alan Turing",
    "History of artificial intelligence",
    "Transformer (deep learning architecture)",
    "GPT-3",
    "AlphaGo",
    "ImageNet",
    "Generative adversarial network",
    "BERT (language model)",
    "Reinforcement learning from human feedback",
    "Large language model",
]

documents = []
for title in article_titles:
    page = wiki.page(title)
    if page.exists():
        documents.append({"title": title, "text": page.text})
        print(f"✓ {title} — {len(page.text):,} characters")
    else:
        print(f"✗ {title} — not found")

print(f"\nTotal articles loaded: {len(documents)}")

## Step 3 — Chunk the Text

In [ ]:
import re
from tqdm.auto import tqdm

CHUNK_SIZE = 5
MIN_TOKENS = 30

def split_into_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if len(s.strip()) > 10]

def make_chunks(sentences, chunk_size):
    return [
        " ".join(sentences[i:i+chunk_size])
        for i in range(0, len(sentences), chunk_size)
    ]

all_chunks = []
for doc in tqdm(documents):
    sentences = split_into_sentences(doc["text"])
    chunks = make_chunks(sentences, CHUNK_SIZE)
    for chunk in chunks:
        token_count = len(chunk) / 4
        if token_count >= MIN_TOKENS:
            all_chunks.append({
                "source": doc["title"],
                "text": chunk,
                "token_count": round(token_count, 1)
            })

print(f"Total chunks: {len(all_chunks)}")

## Step 4 — Embed on GPU

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# GPU embedding — notice device="cuda"
embedding_model = SentenceTransformer("all-mpnet-base-v2", device="cuda")

print("Embedding chunks on GPU...")
texts = [chunk["text"] for chunk in all_chunks]
embeddings = embedding_model.encode(texts, batch_size=64, show_progress_bar=True)

for i, chunk in enumerate(all_chunks):
    chunk["embedding"] = embeddings[i]

print(f"\nEmbedding shape: {embeddings.shape}")
print(f"Each chunk → {embeddings.shape[1]}-dimensional vector")

## Step 5 — Store in LanceDB

In [ ]:
import lancedb
import pyarrow as pa

db = lancedb.connect("ai_history_db_gpu")

data = [
    {
        "source": chunk["source"],
        "text": chunk["text"],
        "token_count": float(chunk["token_count"]),
        "embedding": chunk["embedding"].astype(np.float32).tolist(),
    }
    for chunk in all_chunks
]

schema = pa.schema([
    pa.field("source", pa.string()),
    pa.field("text", pa.string()),
    pa.field("token_count", pa.float64()),
    pa.field("embedding", pa.list_(pa.float32(), list_size=768)),
])

table = db.create_table("ai_history", data=data, schema=schema, mode="overwrite")
table.create_index(metric="cosine", vector_column_name="embedding", index_type="IVF_FLAT")

print(f"Stored {len(data)} chunks in LanceDB")

## Step 6 — Load Qwen2.5-3B on GPU

`n_gpu_layers=-1` offloads ALL model layers to GPU.
Watch the output — you should see `ggml_cuda_init: found 1 CUDA devices`.
If you don't see that, the CUDA build didn't install correctly.

In [ ]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="Qwen/Qwen2.5-3B-Instruct-GGUF",
    filename="*q4_k_m*",
    verbose=True,       # verbose=True so we can see CUDA init confirmation
    n_ctx=4096,
    n_gpu_layers=-1,    # ALL layers on GPU
)

print("\nModel loaded on GPU. Ready to generate.")

## Step 7 — RAG Pipeline

In [ ]:
import textwrap
import time

def retrieve(query, top_k=3):
    query_embedding = embedding_model.encode(query, convert_to_tensor=False)
    results = (
        db["ai_history"]
        .search(query_embedding, vector_column_name="embedding")
        .limit(top_k)
        .to_list()
    )
    return results

def format_prompt(query, context_chunks):
    context = "\n\n".join(
        [f"[Source: {c['source']}]\n{c['text']}" for c in context_chunks]
    )
    return f"""<|im_start|>system
You are a helpful AI historian. Answer questions based only on the provided context.
Be concise, accurate, and cite the source when relevant.<|im_end|>
<|im_start|>user
Context:
{context}

Question: {query}<|im_end|>
<|im_start|>assistant
"""

def ask(query, top_k=3, max_tokens=512):
    print(f"Question: {query}")
    print("=" * 60)

    chunks = retrieve(query, top_k=top_k)
    prompt = format_prompt(query, chunks)

    start = time.time()
    output = llm(
        prompt,
        max_tokens=max_tokens,
        stop=["<|im_end|>", "<|im_start|>"],
        echo=False,
    )
    elapsed = time.time() - start

    answer = output["choices"][0]["text"].strip()
    tokens = len(answer.split())

    print(f"\nAnswer:\n{answer}")
    print(f"\nSources: {list(set(c['source'] for c in chunks))}")
    print(f"Time: {elapsed:.1f}s | Speed: ~{tokens/elapsed:.0f} tokens/sec")
    return answer

## Step 8 — Ask Questions

In [ ]:
ask("What was Alan Turing's contribution to artificial intelligence?")

In [ ]:
ask("Who invented the transformer architecture and when?")

In [ ]:
ask("When did AlphaGo defeat Lee Sedol and why was it significant?")

In [ ]:
ask("What is RLHF and how is it used to train language models?")

In [ ]:
ask("What is the difference between BERT and GPT?")